In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Ensure value columns are numeric where applicable
for df, col in [(metric_df, 'value'), (trace_df, 'value'), (log_df, 'value')]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Helper to aggregate grouped series with required percentiles
def aggregate_series(df, group_cols, value_col):
    g = df.groupby(group_cols)[value_col]
    agg = g.agg(
        count='count',
        mean='mean',
        min='min',
        max='max'
    ).reset_index()
    # Compute percentiles separately to ensure using full series per group
    q50 = g.quantile(0.50).rename('p50').reset_index()
    q90 = g.quantile(0.90).rename('p90').reset_index()
    q95 = g.quantile(0.95).rename('p95').reset_index()
    # Merge percentiles
    agg = agg.merge(q50, on=group_cols, how='left').merge(q90, on=group_cols, how='left').merge(q95, on=group_cols, how='left')
    # Round numeric columns for compactness
    num_cols = ['mean','min','max','p50','p90','p95']
    for c in num_cols:
        if c in agg.columns:
            agg[c] = agg[c].round(6)
    # Sort by count desc and keep top 20 for display (full agg is kept in variable if needed)
    agg_sorted = agg.sort_values('count', ascending=False).reset_index(drop=True)
    return agg_sorted

# 1) metric.csv aggregates
metric_agg = aggregate_series(metric_df, ['cmdb_id','kpi_name'], 'value')
metric_agg_top20 = metric_agg.head(20)
metric_total_rows = metric_df.shape[0]
metric_cmdb_list = pd.Series(metric_df['cmdb_id'].unique()[:50], name='cmdb_id').reset_index(drop=True)

# 2) trace.csv aggregates
trace_agg = aggregate_series(trace_df, ['cmdb_id','trace_name'], 'value')
trace_agg_top20 = trace_agg.head(20)
trace_total_rows = trace_df.shape[0]
trace_tracename_list = pd.Series(trace_df['trace_name'].unique()[:50], name='trace_name').reset_index(drop=True)

# 3) log.csv aggregates
log_agg = aggregate_series(log_df, ['cmdb_id','log_name'], 'value')
log_agg_top20 = log_agg.head(20)
log_total_rows = log_df.shape[0]
log_logname_list = pd.Series(log_df['log_name'].unique()[:50], name='log_name').reset_index(drop=True)

# 4) error_logs.csv summary and earliest 5 samples
error_total_rows = error_df.shape[0]
error_distinct_cmdb = int(error_df['cmdb_id'].nunique()) if 'cmdb_id' in error_df.columns else 0

# Convert timestamp to timezone-aware datetime (if timestamp column exists)
if 'timestamp' in error_df.columns:
    # timestamps are epoch seconds floored to minute; convert using UTC then to Asia/Shanghai
    error_df['_ts_dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # sample earliest 5 by timestamp
    error_samples = error_df.sort_values('timestamp', ascending=True).head(5)[['timestamp','_ts_dt','cmdb_id','message']].copy()
    # present readable timestamp string in Asia/Shanghai
    error_samples['ts_shanghai'] = error_samples['_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    error_samples = error_samples[['timestamp','ts_shanghai','cmdb_id','message']].reset_index(drop=True)
else:
    # fallback empty
    error_samples = pd.DataFrame(columns=['timestamp','ts_shanghai','cmdb_id','message'])

# Compact metadata DataFrame for error logs
error_meta = pd.DataFrame([{
    'total_rows': error_total_rows,
    'distinct_cmdb_count': error_distinct_cmdb,
    'sample_rows_returned': error_samples.shape[0]
}])

# Final displayed variables (compact)
metric_agg_top20, metric_total_rows, metric_cmdb_list, trace_agg_top20, trace_total_rows, trace_tracename_list, log_agg_top20, log_total_rows, log_logname_list, error_meta, error_samples

```
Out[1]:
```
```python
# Summarize the analysis results in plain English as a single compact string.
# Reuse computed variables from prior execution for concrete values.

top_metric = metric_agg_top20.iloc[0] if not metric_agg_top20.empty else None
top_trace = trace_agg_top20.iloc[0] if not trace_agg_top20.empty else None
top_log = log_agg_top20.iloc[0] if not log_agg_top20.empty else None

summary = (
    "Summary of telemetry aggregates:\n"
    f"- metric.csv: total rows = {int(metric_total_rows)}; distinct cmdb_id shown = {int(metric_cmdb_list.shape[0])}.\n"
    f"  Top metric group: {top_metric['cmdb_id']} / {top_metric['kpi_name']} (count={int(top_metric['count'])}, mean={float(top_metric['mean']):.6g}, min={float(top_metric['min']):.6g}, max={float(top_metric['max']):.6g}, p50={float(top_metric['p50']):.6g}, p90={float(top_metric['p90']):.6g}, p95={float(top_metric['p95']):.6g})\n"
    f"  Note: top-20 metric groups are dominated by Mysql02 MySQL KPIs in the sample.\n"
    f"- trace.csv: total rows = {int(trace_total_rows)}; distinct trace_name values shown = {int(trace_tracename_list.shape[0])}.\n"
    f"  Top trace group: {top_trace['cmdb_id']} / {top_trace['trace_name']} (count={int(top_trace['count'])}, mean={float(top_trace['mean']):.6g}, p50={float(top_trace['p50']):.6g}, p95={float(top_trace['p95']):.6g}).\n"
    f"  Note: trace aggregates show IG01/IG02 as primary sources with many to-Tomcat duration and row_count metrics.\n"
    f"- log.csv: total rows = {int(log_total_rows)}; distinct log_name values shown = {int(log_logname_list.shape[0])}.\n"
    f"  Top log group: {top_log['cmdb_id']} / {top_log['log_name']} (count={int(top_log['count'])}, mean={float(top_log['mean']):.6g}, min={float(top_log['min']):.6g}, max={float(top_log['max']):.6g}, p50={float(top_log['p50']):.6g}).\n"
    f"  Note: Tomcat instances have row_count around ~197-208 and error_count is zero in these top groups; apache row_count is larger in the sample.\n"
    f"- error_logs.csv: total rows = {int(error_meta.loc[0,'total_rows'])}; distinct cmdb_id count = {int(error_meta.loc[0,'distinct_cmdb_count'])}. No sample error messages available (0 rows).\n"
    "Overall: metrics and traces are populated and dominated by MySQL and IG->Tomcat interactions respectively; logs show normal row_count activity with negligible error_count in the top groups; no error_log samples were present."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  count          mean        min           max        p50           p90           p95
0   Mysql02                 Mysql-MySQL_3306_Handler Read Prev     31  2.633893e+11   103.1500  5.443379e+11   242.9333  5.443379e+11  5.443379e+11
1   Mysql02                  Mysql-MySQL_3306_Handler Read Rnd     31  5.918262e+05     0.0000  1.223367e+06     1.5932  1.223273e+06  1.223320e+06
2   Mysql02             Mysql-MySQL_3306_Handler Read Rnd Next     31  1.711163e+08    35.6000  3.536897e+08    99.4500  3.536752e+08  3.536824e+08
3   Mysql02                  Mysql-MySQL_3306_Handler Rollback     31  3.227559e+07     0.0000  6.671049e+07    16.9500  6.670744e+07  6.670886e+07
4   Mysql02                 Mysql-MySQL_3306_Handler Savepoint     31  0.000000e+00     0.0000  0.000000e+00     0.0000  0.000000e+00  0.000000e+00
5   Mysql02        Mysql-MySQL_3306_Handler Savepoint Rollback     31  0.000000e+00     0.0000  0.000000e+00     0.0000  0.000000e+00  0.000000e+00
6   Mysql02                    Mysql-MySQL_3306_Handler Update     31  2.111547e+07     8.3000  4.364764e+07    14.4237  4.364496e+07  4.364633e+07
7   Mysql02                Mysql-MySQL_3306_Created Tmp Tables     31  3.371631e+05     0.1333  6.969340e+05     0.2167  6.968950e+05  6.969145e+05
8   Mysql02                 Mysql-MySQL_3306_Handler Read Next     31  2.955885e+07     0.0000  6.108828e+07     0.0000  6.108828e+07  6.108828e+07
9   Mysql02                   Mysql-MySQL_3306_Aborted Clients     31  2.895968e+03     0.0000  5.985000e+03     0.0000  5.985000e+03  5.985000e+03
10  Mysql02                  Mysql-MySQL_3306_Aborted Connects     31  6.580645e+01     0.0000  1.360000e+02     0.0000  1.360000e+02  1.360000e+02
11  Mysql02             Mysql-MySQL_3306_Binlog Cache Disk Use     31  3.629032e+01     0.0000  7.500000e+01     0.0000  7.500000e+01  7.500000e+01
12  Mysql02                  Mysql-MySQL_3306_Binlog Cache Use     31  1.635114e+08    55.7833  3.379954e+08   126.6949  3.379737e+08  3.379849e+08
13  Mysql02        Mysql-MySQL_3306_Binlog stmt cache disk use     31  0.000000e+00     0.0000  0.000000e+00     0.0000  0.000000e+00  0.000000e+00
14  Mysql02              Mysql-MySQL_3306_Innodb Row Lock Time     31  6.745326e+06     0.4500  1.395794e+07   112.1667  1.393957e+07  1.394651e+07
15  Mysql02             Mysql-MySQL_3306_Innodb Row Lock Waits     31  4.137375e+05     0.0678  8.553990e+05     1.3667  8.551610e+05  8.552745e+05
16  Mysql02    Mysql-MySQL_3306_Innodb buffer pool pages dirty     31  4.548387e+03  2191.0000  5.943000e+03  4748.0000  5.525000e+03  5.780500e+03
17  Mysql02  Mysql-MySQL_3306_Innodb buffer pool pages flushed     31  3.097214e+07     0.0000  6.401284e+07    14.4667  6.401169e+07  6.401234e+07
18  Mysql02     Mysql-MySQL_3306_Innodb buffer pool pages free     31  8.190000e+03  8185.0000  8.192000e+03  8190.0000  8.192000e+03  8.192000e+03
19  Mysql02     Mysql-MySQL_3306_Innodb buffer pool pages misc     31  3.052961e+01     0.0167  8.400000e+01     0.0339  7.800000e+01  8.150000e+01, 29584, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
21    Tomcat03
22    Tomcat04
23    apache01
24    apache02
25    dockerA2
Name: cmdb_id, Length: 26, dtype: object,    cmdb_id                       trace_name  count        mean        min          max         p50         p90         p95
0     IG01         trace.self.duration_mean     31    0.332930   0.101610     2.649364    0.225262    0.407210    0.589135
1     IG01          trace.self.duration_p95     31    1.666640   1.020000    10.011900    1.087000    2.178000    2.770000
2     IG01             trace.self.row_count     31  787.612903  22.000000  1070.000000  800.000000  946.000000  984.000000
3     IG01  trace.to_Tomcat01.duration_mean     31    0.360030   0.057941     3.513667    0.241413    0.436356    0.525307
4     IG01   trace.to_Tomcat01.duration_p95     31    1.587485   0.200600     9.246300    1.077000    2.316900    2.362500
5     IG01      trace.to_Tomcat01.row_count     31   97.741935   3.000000   138.000000  100.000000  119.000000  124.500000
6     IG01  trace.to_Tomcat02.duration_mean     31    0.319644   0.089804     2.553200    0.230839    0.423448    0.497624
7     IG01   trace.to_Tomcat02.duration_p95     31    1.477392   0.554750     5.595800    1.100100    2.064700    2.409450
8     IG01      trace.to_Tomcat02.row_count     31   98.967742   5.000000   139.000000  103.000000  122.000000  123.500000
9     IG01  trace.to_Tomcat03.duration_mean     31    0.400186   0.089505     4.560000    0.211115    0.399323    0.696065
10    IG01   trace.to_Tomcat03.duration_p95     31    1.456818   0.564900     4.560000    1.077100    2.249500    3.341750
11    IG01      trace.to_Tomcat03.row_count     31   96.870968   1.000000   133.000000   99.000000  119.000000  126.000000
12    IG01  trace.to_Tomcat04.duration_mean     31    0.270353   0.099067     0.874427    0.228162    0.444639    0.539513
13    IG01   trace.to_Tomcat04.duration_p95     31    1.458071   1.015700     5.483000    1.084550    2.198500    2.563925
14    IG01      trace.to_Tomcat04.row_count     31  100.225806   2.000000   140.000000  103.000000  117.000000  133.000000
15    IG02         trace.self.duration_mean     31    0.341265   0.110438     3.228885    0.217739    0.383867    0.555913
16    IG02          trace.self.duration_p95     31    1.544337   1.021000     9.052500    1.085000    2.100450    2.503000
17    IG02             trace.self.row_count     31  800.000000  26.000000  1016.000000  826.000000  954.000000  985.000000
18    IG02  trace.to_Tomcat01.duration_mean     31    0.466564   0.068276     7.428500    0.210221    0.430472    0.504130
19    IG02   trace.to_Tomcat01.duration_p95     31    1.524635   0.161300     9.754550    1.074200    2.268100    2.446200, 7026, 0            trace.self.duration_mean
1             trace.self.duration_p95
2                trace.self.row_count
3     trace.to_Tomcat01.duration_mean
4      trace.to_Tomcat01.duration_p95
                   ...               
45    trace.to_dockerA2.duration_mean
46     trace.to_dockerA2.duration_p95
47        trace.to_dockerA2.row_count
48    trace.to_dockerB1.duration_mean
49     trace.to_dockerB1.duration_p95
Name: trace_name, Length: 50, dtype: object,      cmdb_id          log_name  count        mean    min     max      p50      p90       p95
0   Tomcat01   log.error_count     31    0.000000   0.00    0.00    0.000    0.000    0.0000
1   Tomcat01     log.row_count     31  196.806452   2.00  255.00  202.000  235.000  237.5000
2   Tomcat02   log.error_count     31    0.000000   0.00    0.00    0.000    0.000    0.0000
3   Tomcat02     log.row_count     31  199.483871   8.00  273.00  208.000  236.000  243.5000
4   Tomcat03   log.error_count     31    0.000000   0.00    0.00    0.000    0.000    0.0000
5   Tomcat04     log.row_count     31  199.193548   3.00  262.00  206.000  239.000  259.5000
6   Tomcat04   log.error_count     31    0.000000   0.00    0.00    0.000    0.000    0.0000
7   Tomcat03     log.row_count     31  198.419355   1.00  253.00  205.000  247.000  249.5000
8   apache02   log.error_count     31    0.000000   0.00    0.00    0.000    0.000    0.0000
9   apache02     log.row_count     31  465.677419  12.00  827.00  449.000  719.000  750.5000
10  apache01   log.error_count     27    0.000000   0.00    0.00    0.000    0.000    0.0000
11  apache01     log.row_count     27  376.740741  12.00  521.00  412.000  470.600  488.6000
12  Tomcat04  log.gc_pause_max     11    0.066364   0.02    0.10    0.080    0.090    0.0950
13  Tomcat02  log.gc_pause_max     10    0.058000   0.02    0.10    0.075    0.091    0.0955
14  Tomcat03  log.gc_pause_max     10    0.059000   0.02    0.10    0.065    0.091    0.0955
15  Tomcat01  log.gc_pause_max      9    0.050000   0.02    0.09    0.040    0.082    0.0860, 404, 0     log.error_count
1       log.row_count
2    log.gc_pause_max
Name: log_name, dtype: object,    total_rows  distinct_cmdb_count  sample_rows_returned
0           0                    0                     0, Empty DataFrame
Columns: [timestamp, ts_shanghai, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel: metric_df, trace_df, log_df, tz
# If any are missing, load them (defensive)
try:
    metric_df
    trace_df
    log_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    trace_df = pd.read_csv('trace.csv')
    log_df = pd.read_csv('log.csv')

# Ensure numeric value columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = pd.Timestamp('2021-03-25 20:30:00', tz=tz).tz_convert('UTC').timestamp()
end_ts   = pd.Timestamp('2021-03-25 21:00:00', tz=tz).tz_convert('UTC').timestamp()
start_epoch = int(start_ts)
end_epoch = int(end_ts)

# Allowed root-cause components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Helper to compute global p05/p95 thresholds per group (before filtering)
def compute_thresholds(df, group_cols, value_col):
    g = df.groupby(group_cols)[value_col]
    p05 = g.quantile(0.05).rename('p05').reset_index()
    p95 = g.quantile(0.95).rename('p95').reset_index()
    thresh = p05.merge(p95, on=group_cols, how='outer')
    # keep numeric precision reasonable
    thresh['p05'] = thresh['p05'].astype(float)
    thresh['p95'] = thresh['p95'].astype(float)
    return thresh

# Compute thresholds
metric_thresh = compute_thresholds(metric_df, ['cmdb_id','kpi_name'], 'value')
trace_thresh  = compute_thresholds(trace_df,  ['cmdb_id','trace_name'], 'value')
log_thresh    = compute_thresholds(log_df,    ['cmdb_id','log_name'], 'value')

# Filter data to allowed components and incident window, store for later use
incident_metric_df = metric_df[
    metric_df['cmdb_id'].isin(allowed) &
    (metric_df['timestamp'].between(start_epoch, end_epoch))
].copy()

incident_trace_df = trace_df[
    trace_df['cmdb_id'].isin(allowed) &
    (trace_df['timestamp'].between(start_epoch, end_epoch))
].copy()

incident_log_df = log_df[
    log_df['cmdb_id'].isin(allowed) &
    (log_df['timestamp'].between(start_epoch, end_epoch))
].copy()

# Function to analyze anomalies for a file
def analyze_anomalies(incident_df, thresh_df, group_cols, name_col):
    if incident_df.empty:
        # empty summary
        empty_cols = ['component','metric_name','p95','p05','anomaly_count',
                      'earliest_anomaly_timestamp','max_value_in_window',
                      'min_value_in_window','max_deviation_ratio']
        return pd.DataFrame(columns=empty_cols), 0

    # Merge thresholds into incident rows
    merged = incident_df.merge(thresh_df, left_on=group_cols, right_on=group_cols, how='left')

    # Determine anomaly flags (global thresholds computed before filtering)
    # High: value >= p95; Low: value <= p05
    merged['is_high'] = merged['value'] >= merged['p95']
    merged['is_low']  = merged['value'] <= merged['p05']
    merged['is_anomaly'] = merged['is_high'] | merged['is_low']

    # For earliest anomaly timestamp convert to Asia/Shanghai formatted string later
    def compute_group_stats(g):
        comp = g.name[0]
        mname = g.name[1]
        p95 = float(g['p95'].iloc[0]) if pd.notna(g['p95'].iloc[0]) else np.nan
        p05 = float(g['p05'].iloc[0]) if pd.notna(g['p05'].iloc[0]) else np.nan

        anomaly_count = int(g['is_anomaly'].sum())
        if anomaly_count > 0:
            earliest_ts = int(g.loc[g['is_anomaly'], 'timestamp'].min())
            # convert earliest_ts to Asia/Shanghai string
            earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz)
            earliest_str = earliest_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            earliest_ts = np.nan
            earliest_str = None

        max_val = float(g['value'].max()) if g['value'].notna().any() else np.nan
        min_val = float(g['value'].min()) if g['value'].notna().any() else np.nan

        # Compute deviation ratios safely
        high_ratio = np.nan
        low_ratio = np.nan
        # If there are high anomalies, compute max_value/p95 (handle p95==0)
        if g['is_high'].any() and (pd.notna(p95)):
            if p95 == 0:
                high_ratio = np.inf
            else:
                high_ratio = max_val / p95
        # If low anomalies exist, compute p05/min_value (handle min_val==0)
        if g['is_low'].any() and (pd.notna(p05)):
            if min_val == 0:
                low_ratio = np.inf
            else:
                low_ratio = p05 / min_val
        # choose larger ratio (treat NaN as -inf)
        ratios = [r for r in [high_ratio, low_ratio] if not (pd.isna(r))]
        max_dev = float(max(ratios)) if ratios else np.nan

        return pd.Series({
            'component': comp,
            'metric_name': mname,
            'p95': round(p95, 6) if pd.notna(p95) else np.nan,
            'p05': round(p05, 6) if pd.notna(p05) else np.nan,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_str,
            'max_value_in_window': round(max_val, 6) if pd.notna(max_val) else np.nan,
            'min_value_in_window': round(min_val, 6) if pd.notna(min_val) else np.nan,
            'max_deviation_ratio': round(max_dev, 6) if pd.notna(max_dev) else np.nan
        })

    grouped = merged.groupby(group_cols, sort=False)
    summary = grouped.apply(compute_group_stats).reset_index(drop=True)

    # Sort by anomaly_count desc and keep top 20
    summary_sorted = summary.sort_values('anomaly_count', ascending=False).reset_index(drop=True)
    top20 = summary_sorted.head(20)

    # Total anomalies (count of anomaly rows in incident window)
    total_anomalies = int(merged['is_anomaly'].sum())

    return top20, total_anomalies

# Analyze metric, trace, log
metric_summary_top20, metric_total_anomalies = analyze_anomalies(
    incident_metric_df, metric_thresh, ['cmdb_id','kpi_name'], 'kpi_name'
)

trace_summary_top20, trace_total_anomalies = analyze_anomalies(
    incident_trace_df, trace_thresh, ['cmdb_id','trace_name'], 'trace_name'
)

log_summary_top20, log_total_anomalies = analyze_anomalies(
    incident_log_df, log_thresh, ['cmdb_id','log_name'], 'log_name'
)

# Compact outputs: top20 summaries and totals and shapes of incident dfs
metric_summary_top20, metric_total_anomalies, incident_metric_df.shape, trace_summary_top20, trace_total_anomalies, incident_trace_df.shape, log_summary_top20, log_total_anomalies, incident_log_df.shape

```
Out[2]:
```
```python
# Summarize the incident-window anomaly analysis as a single plain-English string.
# Reuse previously computed kernel variables: metric_total_anomalies, trace_total_anomalies, log_total_anomalies,
# metric_summary_top20, trace_summary_top20, log_summary_top20

# Defensive checks for variable existence
try:
    metric_total_anomalies
except NameError:
    metric_total_anomalies = 0
try:
    trace_total_anomalies
except NameError:
    trace_total_anomalies = 0
try:
    log_total_anomalies
except NameError:
    log_total_anomalies = 0

def top_info(df):
    if df is None or df.empty:
        return None
    row = df.iloc[0]
    return f"{row['component']} / {row['metric_name']} (anomalies={int(row['anomaly_count'])})"

metric_top = top_info(metric_summary_top20)
trace_top = top_info(trace_summary_top20)
log_top = top_info(log_summary_top20)

summary = (
    "Incident-window anomaly summary (2021-03-25 20:30–21:00 UTC+8) for allowed components:\n"
    f"- Totals: metric anomalies = {int(metric_total_anomalies)}, trace anomalies = {int(trace_total_anomalies)}, log anomalies = {int(log_total_anomalies)}.\n"
    f"- Metrics: top anomalous group: {metric_top or 'none'}. Many metric anomalies originate from MySQL/MG/IG related KPIs; several metric groups show p95 or p05 equal to zero causing infinite deviation ratios.\n"
    f"- Traces: top anomalous group: {trace_top or 'none'}. Trace anomalies concentrate on row_count and duration metrics between MG/IG and Tomcat nodes (notably MG01, MG02, IG01/IG02, Tomcat01/04), with some large deviation ratios indicating spikes in counts/durations.\n"
    f"- Logs: top anomalous group: {log_top or 'none'}. Numerous log.error_count entries are flagged (p95/p05 are zero so deviation ratio is infinite); row_count anomalies appear for Tomcat and apache instances (Tomcat01/02/03/04 and apache01/apache02).\n"
    "Overall conclusion: The incident window shows heavy anomaly activity across metrics (especially MySQL/MG/IG KPIs), elevated trace row_count/duration spikes between IG/MG and Tomcat, and many log error_count zero-threshold anomalies. Earliest anomalies appear at the window start (around 2021-03-25 20:30:00 UTC+8)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                        metric_name           p95           p05  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  max_deviation_ratio
0       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.886139e+07  5.886139e+07             31    2021-03-25 20:30:00 CST         5.886139e+07         5.886139e+07                  1.0
1       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07  5.846119e+07             31    2021-03-25 20:30:00 CST         5.846119e+07         5.846119e+07                  1.0
2       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         1.000000e+00         0.000000e+00                  inf
3       MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         0.000000e+00         0.000000e+00                  inf
4       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09             31    2021-03-25 20:30:00 CST         2.040136e+09         2.040136e+09                  1.0
5    Mysql02              Mysql-MySQL_3306_LongestTrxActiveTime  5.000000e-01  0.000000e+00             31    2021-03-25 20:30:00 CST         1.000000e+00         0.000000e+00                  inf
6    Mysql02                Mysql-MySQL_3306_MaxTrxRowsModified  1.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         1.000000e+00         0.000000e+00                  inf
7    Mysql02          Mysql-MySQL_3306_Opened table definitions  5.716000e+03  0.000000e+00             31    2021-03-25 20:30:00 CST         5.716000e+03         0.000000e+00                  inf
8    Mysql02                Mysql-MySQL_3306_Qcache Free Blocks  1.000000e+00  1.000000e+00             31    2021-03-25 20:30:00 CST         1.000000e+00         1.000000e+00                  1.0
9    Mysql02                Mysql-MySQL_3306_Qcache Free Memory  1.031832e+06  1.031832e+06             31    2021-03-25 20:30:00 CST         1.031832e+06         1.031832e+06                  1.0
10   Mysql02                       Mysql-MySQL_3306_Qcache Hits  0.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         0.000000e+00         0.000000e+00                  inf
11   Mysql02                    Mysql-MySQL_3306_Qcache Inserts  0.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         0.000000e+00         0.000000e+00                  inf
12   Mysql02                 Mysql-MySQL_3306_Tc log page waits  0.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         0.000000e+00         0.000000e+00                  inf
13   Mysql02                    Mysql-MySQL_3306_Threads Cached  1.500000e+01  1.400000e+01             31    2021-03-25 20:30:00 CST         1.500000e+01         1.400000e+01                  1.0
14   Mysql02                   Mysql-MySQL_3306_Threads Created  3.700000e+01  0.000000e+00             31    2021-03-25 20:30:00 CST         3.700000e+01         0.000000e+00                  inf
15   Mysql02                  Mysql-MySQL_3306_ThreadsConnected  2.300000e+01  2.200000e+01             31    2021-03-25 20:30:00 CST         2.300000e+01         2.200000e+01                  1.0
16   Mysql02              Mysql-MySQL_3306_Max Used Connections  3.700000e+01  3.700000e+01             31    2021-03-25 20:30:00 CST         3.700000e+01         3.700000e+01                  1.0
17   Mysql02               Mysql-MySQL_3306_Max trx rows locked  1.000000e+00  0.000000e+00             31    2021-03-25 20:30:00 CST         1.000000e+00         0.000000e+00                  inf
18   Mysql02                    Mysql-MySQL_3306_MaxConnections  2.000000e+03  2.000000e+03             31    2021-03-25 20:30:00 CST         2.000000e+03         2.000000e+03                  1.0
19  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1.051000e+03  1.051000e+03             31    2021-03-25 20:30:00 CST         1.051000e+03         1.051000e+03                  1.0, 19283, (28165, 4),    component                       metric_name         p95         p05  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  max_deviation_ratio
0       MG01     trace.from_dockerA1.row_count   25.100000    4.000000              7    2021-03-25 20:30:00 CST            33.000000             4.000000             1.314741
1   Tomcat01           trace.self.duration_p95    0.212675    0.023000              7    2021-03-25 20:31:00 CST             3.978100             0.021000            18.705066
2       MG02     trace.from_dockerA2.row_count   29.650000    5.000000              6    2021-03-25 20:36:00 CST            31.000000             2.000000             2.500000
3   Tomcat01           trace.to_MG01.row_count  120.000000   79.500000              6    2021-03-25 20:34:00 CST           127.000000             3.000000            26.500000
4       MG01     trace.from_Tomcat01.row_count  120.000000   79.500000              6    2021-03-25 20:34:00 CST           127.000000             3.000000            26.500000
5       IG01       trace.to_Tomcat04.row_count  133.000000   78.000000              5    2021-03-25 20:34:00 CST           140.000000             2.000000            39.000000
6       MG01     trace.from_dockerB1.row_count  230.000000   54.500000              5    2021-03-25 20:37:00 CST           279.000000             1.000000            54.500000
7       MG01       trace.to_dockerB2.row_count   62.200000   36.000000              5    2021-03-25 20:33:00 CST            71.000000            27.000000             1.333333
8       MG02     trace.from_dockerB2.row_count   76.000000   18.800000              5    2021-03-25 20:37:00 CST            80.000000            12.000000             1.566667
9       MG02     trace.from_dockerA1.row_count   26.500000    5.000000              5    2021-03-25 20:30:00 CST            34.000000             1.000000             5.000000
10      MG01  trace.from_dockerB1.duration_p95    1.258850    0.010000              5    2021-03-25 20:43:00 CST             4.516000             0.009600             3.587401
11  Tomcat04           trace.self.duration_p95    0.243750    0.023000              5    2021-03-25 20:40:00 CST             0.333500             0.023000             1.368205
12  Tomcat04         trace.from_IG01.row_count  133.000000   78.000000              5    2021-03-25 20:34:00 CST           140.000000             2.000000            39.000000
13      IG01              trace.self.row_count  984.000000  575.000000              4    2021-03-25 20:34:00 CST          1070.000000            22.000000            26.136364
14      IG01    trace.to_Tomcat04.duration_p95    2.563925    1.024600              4    2021-03-25 20:44:00 CST             5.483000             1.015700             2.138518
15      IG01   trace.to_Tomcat04.duration_mean    0.539513    0.116724              4    2021-03-25 20:44:00 CST             0.874427             0.099067             1.620771
16      IG01       trace.to_Tomcat03.row_count  126.000000   70.000000              4    2021-03-25 20:38:00 CST           133.000000             1.000000            70.000000
17      IG02          trace.self.duration_mean    0.555913    0.122559              4    2021-03-25 20:43:00 CST             3.228885             0.110438             5.808256
18      IG02   trace.to_Tomcat01.duration_mean    0.504130    0.102012              4    2021-03-25 20:43:00 CST             7.428500             0.068276            14.735287
19      IG02    trace.to_Tomcat01.duration_p95    2.446200    1.015750              4    2021-03-25 20:40:00 CST             9.754550             0.161300             6.297272, 698, (5181, 4),    component       metric_name       p95       p05  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  max_deviation_ratio
0   Tomcat01   log.error_count    0.0000    0.0000             31    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
1   Tomcat02   log.error_count    0.0000    0.0000             31    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
2   Tomcat04   log.error_count    0.0000    0.0000             31    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
3   Tomcat03   log.error_count    0.0000    0.0000             31    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
4   apache02   log.error_count    0.0000    0.0000             31    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
5   apache01   log.error_count    0.0000    0.0000             27    2021-03-25 20:30:00 CST                 0.00                 0.00                  inf
6   Tomcat02  log.gc_pause_max    0.0955    0.0200              5    2021-03-25 20:33:00 CST                 0.10                 0.02             1.047120
7   Tomcat01     log.row_count  237.5000  153.5000              4    2021-03-25 20:34:00 CST               255.00                 2.00            76.750000
8   Tomcat03     log.row_count  249.5000  136.5000              4    2021-03-25 20:34:00 CST               253.00                 1.00           136.500000
9   Tomcat02     log.row_count  243.5000  135.0000              4    2021-03-25 20:33:00 CST               273.00                 8.00            16.875000
10  apache01     log.row_count  488.6000   78.3000              4    2021-03-25 20:33:00 CST               521.00                12.00             6.525000
11  Tomcat04     log.row_count  259.5000  145.5000              4    2021-03-25 20:34:00 CST               262.00                 3.00            48.500000
12  apache02     log.row_count  750.5000  338.0000              4    2021-03-25 20:49:00 CST               827.00                12.00            28.166667
13  Tomcat01  log.gc_pause_max    0.0860    0.0200              3    2021-03-25 20:32:00 CST                 0.09                 0.02             1.046512
14  Tomcat04  log.gc_pause_max    0.0950    0.0250              2    2021-03-25 20:38:00 CST                 0.10                 0.02             1.250000
15  Tomcat03  log.gc_pause_max    0.0955    0.0245              2    2021-03-25 20:42:00 CST                 0.10                 0.02             1.225000, 218, (404, 4))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables if present
try:
    incident_metric_df
    metric_thresh
    tz
except NameError:
    # defensive loading if missing
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    incident_metric_df = metric_df.copy()  # fallback: full data
    # compute thresholds globally
    metric_thresh = metric_df.groupby(['cmdb_id','kpi_name'])['value'].quantile([0.05,0.95]).unstack(level=-1).reset_index().rename(columns={0.05:'p05',0.95:'p95'})

# Merge global thresholds into the incident window metric rows
merged = incident_metric_df.merge(metric_thresh, on=['cmdb_id','kpi_name'], how='left')

# Flag anomalies using global p95/p05
merged['is_high'] = merged['value'] >= merged['p95']
merged['is_low']  = merged['value'] <= merged['p05']
merged['is_anomaly'] = merged['is_high'] | merged['is_low']

# Keep only anomaly rows
anoms = merged[merged['is_anomaly']].copy()

# If no anomaly rows, prepare empty results
if anoms.empty:
    metric_faults_df = pd.DataFrame(columns=[
        'component','metric_name','fault_start_timestamp','fault_start_shanghai',
        'duration_minutes','anomaly_points_count','first_value','max_value','min_value',
        'global_p95','global_p05','severity_ratio'
    ])
    metric_faults_by_component_df = pd.DataFrame(columns=['component','fault_count'])
else:
    # Sort and compute sequence ids for consecutive timestamps within each (cmdb_id,kpi_name)
    anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # For each group, compute diff and seq id
    anoms['time_diff'] = anoms.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(0)
    # start new sequence whenever gap > 60 seconds
    anoms['seq_break'] = (anoms['time_diff'] > 60).astype(int)
    # cumulative sequence id per group
    anoms['seq_id'] = anoms.groupby(['cmdb_id','kpi_name'])['seq_break'].cumsum()

    # Now aggregate each sequence
    grp_cols = ['cmdb_id','kpi_name','seq_id']
    seq_agg = anoms.groupby(grp_cols).agg(
        anomaly_points_count=('timestamp','size'),
        fault_start_timestamp=('timestamp','min'),
        first_idx=('timestamp','idxmin'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        global_p95=('p95','first'),
        global_p05=('p05','first'),
        any_high=('is_high','any'),
        any_low=('is_low','any')
    ).reset_index()

    # Filter subsequences with at least 2 points
    seq_agg = seq_agg[seq_agg['anomaly_points_count'] >= 2].copy()

    # If still empty after filtering
    if seq_agg.empty:
        metric_faults_df = pd.DataFrame(columns=[
            'component','metric_name','fault_start_timestamp','fault_start_shanghai',
            'duration_minutes','anomaly_points_count','first_value','max_value','min_value',
            'global_p95','global_p05','severity_ratio'
        ])
        metric_faults_by_component_df = pd.DataFrame(columns=['component','fault_count'])
    else:
        # Compute additional fields: duration_minutes (same as anomaly_points_count), first_value using first_idx
        # Map first_value from anoms by index
        first_values = anoms.loc[seq_agg['first_idx'], 'value'].values
        seq_agg['first_value'] = first_values

        seq_agg['duration_minutes'] = seq_agg['anomaly_points_count']  # timestamps are per-minute points

        # Compute severity ratios
        def compute_severity(row):
            max_val = row['max_value_in_window']
            min_val = row['min_value_in_window']
            p95 = row['global_p95']
            p05 = row['global_p05']

            high_ratio = np.nan
            low_ratio = np.nan

            if row['any_high']:
                if pd.notna(p95):
                    if p95 == 0:
                        high_ratio = np.inf
                    else:
                        high_ratio = max_val / p95
                else:
                    high_ratio = np.nan

            if row['any_low']:
                if pd.notna(p05):
                    if min_val == 0:
                        low_ratio = np.inf
                    else:
                        low_ratio = p05 / min_val
                else:
                    low_ratio = np.nan

            # If neither computed, fallback to max_val/p95 if possible
            candidates = [r for r in [high_ratio, low_ratio] if not pd.isna(r)]
            if not candidates:
                if pd.notna(p95) and p95 != 0:
                    return max_val / p95
                elif pd.notna(p95) and p95 == 0:
                    return np.inf
                else:
                    return np.nan
            return max(candidates)

        seq_agg['severity_ratio'] = seq_agg.apply(compute_severity, axis=1)

        # Convert fault_start_timestamp to Asia/Shanghai human datetime string
        seq_agg['fault_start_shanghai'] = pd.to_datetime(seq_agg['fault_start_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

        # Assemble final columns and names
        metric_faults_df = seq_agg.rename(columns={
            'cmdb_id':'component',
            'kpi_name':'metric_name',
            'max_value_in_window':'max_value',
            'min_value_in_window':'min_value',
            'global_p95':'global_p95',
            'global_p05':'global_p05',
            'fault_start_timestamp':'fault_start_timestamp'
        })[[
            'component','metric_name','fault_start_timestamp','fault_start_shanghai',
            'duration_minutes','anomaly_points_count','first_value','max_value','min_value',
            'global_p95','global_p05','severity_ratio'
        ]].copy()

        # Filter candidate faults to severity_ratio > 1.5
        metric_faults_df = metric_faults_df[metric_faults_df['severity_ratio'] > 1.5].copy()

        # Sort by severity_ratio desc then earliest fault_start_timestamp
        metric_faults_df = metric_faults_df.sort_values(['severity_ratio','fault_start_timestamp'], ascending=[False, True]).reset_index(drop=True)

        # Limit to top 20
        metric_faults_df = metric_faults_df.head(20).reset_index(drop=True)

        # Component-level aggregate: count of faults per component (top 10)
        metric_faults_by_component_df = metric_faults_df.groupby('component').size().reset_index(name='fault_count').sort_values('fault_count', ascending=False).head(10).reset_index(drop=True)

# Final compact outputs (top faults and component summary)
metric_faults_df, metric_faults_by_component_df

```
Out[3]:
```
```python
# Summarize metric fault identification results in plain English as a single string.
# Reuse kernel variables metric_faults_df and metric_faults_by_component_df.

try:
    total_faults = int(metric_faults_df.shape[0])
    comp_summary_df = metric_faults_by_component_df
    earliest_start = metric_faults_df['fault_start_shanghai'].min() if total_faults > 0 else None
    # top components as "component(count)" list
    top_comps = ", ".join(f"{row.component}({int(row.fault_count)})" for row in comp_summary_df.itertuples(index=False)) if not comp_summary_df.empty else "none"
    # sample top 3 faults by severity
    sample_faults = []
    if total_faults > 0:
        sample_rows = metric_faults_df.sort_values(['severity_ratio','fault_start_timestamp'], ascending=[False, True]).head(3)
        for r in sample_rows.itertuples(index=False):
            sample_faults.append(f"{r.component} / {r.metric_name} (severity={r.severity_ratio}, duration={r.duration_minutes}m)")
    sample_text = "; ".join(sample_faults) if sample_faults else "none"

    summary = (
        f"Metric fault identification (incident window 2021-03-25 20:30–21:00 UTC+8):\n"
        f"- Total candidate faults (severity > 1.5, consecutive >=2 mins): {total_faults}.\n"
        f"- Top components by fault count: {top_comps}.\n"
        f"- Earliest detected fault start: {earliest_start}.\n"
        f"- Notes: Many faults are on Mysql01 (16 faults) and MG nodes (MG01, MG02)."
        " Several faults span the entire window (duration 31 minutes) and many severity ratios are infinite due to global p95/p05 being zero for those metrics.\n"
        f"- Example top faults: {sample_text}.\n"
        "Recommendation: prioritize Mysql01 metrics (multiple MySQL KPIs) and MG nodes for follow-up; verify zero-threshold cases that produce infinite severity ratios."
    )
except Exception as e:
    summary = f"Summary unavailable due to error: {e}"

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   component                                     metric_name  fault_start_timestamp     fault_start_shanghai  duration_minutes  anomaly_points_count  first_value   max_value  min_value  global_p95  global_p05  severity_ratio
0       MG01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
1       MG01    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         1.0        0.0         0.0         0.0             inf
2       MG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
3       MG02    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         1.0        0.0         0.0         0.0             inf
4    Mysql01                Mysql-MySQL_3306_Aborted Clients             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0      6902.0        0.0      6902.0         0.0             inf
5    Mysql01               Mysql-MySQL_3306_Aborted Connects             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0     37918.0        0.0     37918.0         0.0             inf
6    Mysql01          Mysql-MySQL_3306_Binlog Cache Disk Use             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0        59.0        0.0        59.0         0.0             inf
7    Mysql01     Mysql-MySQL_3306_Binlog stmt cache disk use             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
8    Mysql01                     Mysql-MySQL_3306_Com Delete             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         1.0        0.0         1.0         0.0             inf
9    Mysql01               Mysql-MySQL_3306_Com Delete Multi             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
10   Mysql01              Mysql-MySQL_3306_Com Insert Select             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
11   Mysql01                       Mysql-MySQL_3306_Com Load             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
12   Mysql01                    Mysql-MySQL_3306_Com Replace             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
13   Mysql01             Mysql-MySQL_3306_Com Replace Select             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
14   Mysql01               Mysql-MySQL_3306_Com Update Multi             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
15   Mysql01              Mysql-MySQL_3306_Created Tmp Files             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0        27.0        0.0        27.0         0.0             inf
16   Mysql01       Mysql-MySQL_3306_CurrentSQLMaxRunningTime             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0         0.0        0.0         0.0         0.0             inf
17   Mysql01                 Mysql-MySQL_3306_Handler Delete             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0        11.0        0.0        11.0         0.0             inf
18   Mysql01              Mysql-MySQL_3306_Handler Read Next             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0  12752868.0        0.0  12752868.0         0.0             inf
19   Mysql01              Mysql-MySQL_3306_Handler Read Prev             1616675400  2021-03-25 20:30:00 CST                31                    31          0.0       722.0        0.0       722.0         0.0             inf,   component  fault_count
0   Mysql01           16
1      MG01            2
2      MG02            2)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing kernel variables: metric_faults_df, incident_trace_df, trace_thresh, tz
# Defensive checks
try:
    metric_faults_df
except NameError:
    metric_faults_df = pd.DataFrame(columns=[
        'component','metric_name','fault_start_timestamp','fault_start_shanghai',
        'duration_minutes','anomaly_points_count','first_value','max_value','min_value',
        'global_p95','global_p05','severity_ratio'
    ])
try:
    incident_trace_df
except NameError:
    incident_trace_df = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
try:
    trace_thresh
except NameError:
    trace_thresh = pd.DataFrame(columns=['cmdb_id','trace_name','p05','p95'])
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) metric faults earliest per component
if metric_faults_df.empty:
    metric_fault_earliest_df = pd.DataFrame(columns=['component','earliest_timestamp','earliest_shanghai','count'])
else:
    # find earliest fault per component and fault counts
    # get index of earliest fault_start_timestamp per component
    idx = metric_faults_df.groupby('component')['fault_start_timestamp'].idxmin()
    earliest = metric_faults_df.loc[idx, ['component','fault_start_timestamp','fault_start_shanghai']].copy()
    counts = metric_faults_df.groupby('component').size().reset_index(name='count')
    metric_fault_earliest_df = earliest.merge(counts, on='component', how='left')
    metric_fault_earliest_df = metric_fault_earliest_df.rename(columns={
        'fault_start_timestamp':'earliest_timestamp',
        'fault_start_shanghai':'earliest_shanghai'
    })
    metric_fault_earliest_df = metric_fault_earliest_df.sort_values('earliest_timestamp', ascending=True).head(10).reset_index(drop=True)

# 2) trace anomalies earliest per component
# Merge incident_trace_df with global thresholds and flag anomalies (use full-series thresholds already computed)
if incident_trace_df.empty or trace_thresh.empty:
    trace_earliest_df = pd.DataFrame(columns=['component','earliest_timestamp','earliest_shanghai','count'])
else:
    merged_trace = incident_trace_df.merge(trace_thresh, on=['cmdb_id','trace_name'], how='left')
    merged_trace['is_high'] = merged_trace['value'] >= merged_trace['p95']
    merged_trace['is_low']  = merged_trace['value'] <= merged_trace['p05']
    merged_trace['is_anomaly'] = merged_trace['is_high'] | merged_trace['is_low']

    anomalies_trace = merged_trace[merged_trace['is_anomaly']].copy()
    if anomalies_trace.empty:
        trace_earliest_df = pd.DataFrame(columns=['component','earliest_timestamp','earliest_shanghai','count'])
    else:
        agg = anomalies_trace.groupby('cmdb_id').agg(
            earliest_timestamp=('timestamp','min'),
            count=('timestamp','size')
        ).reset_index().rename(columns={'cmdb_id':'component'})
        # convert earliest_timestamp to Asia/Shanghai string
        agg['earliest_shanghai'] = pd.to_datetime(agg['earliest_timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
        trace_earliest_df = agg[['component','earliest_timestamp','earliest_shanghai','count']].sort_values('earliest_timestamp', ascending=True).head(10).reset_index(drop=True)

# Display compact results (stored in kernel variables for later steps)
metric_fault_earliest_df, trace_earliest_df

```
Out[4]:
```
```python
# Create a concise plain-English summary using previously computed kernel tables:
# metric_fault_earliest_df and trace_earliest_df

try:
    mfe = metric_fault_earliest_df
    te = trace_earliest_df
except NameError:
    mfe = pd.DataFrame(columns=['component','earliest_timestamp','earliest_shanghai','count'])
    te = pd.DataFrame(columns=['component','earliest_timestamp','earliest_shanghai','count'])

parts = []

# Metric faults earliest per component
if mfe.empty:
    parts.append("Metric faults: none detected in metric_faults_df.")
else:
    rows = []
    for r in mfe.itertuples(index=False):
        rows.append(f"{r.component} earliest {r.earliest_shanghai} (faults={int(r.count)})")
    parts.append("Metric faults earliest per component: " + "; ".join(rows) + ".")

# Trace anomalies earliest per component
if te.empty:
    parts.append("Trace anomalies: none detected in incident trace anomalies.")
else:
    rows = []
    for r in te.itertuples(index=False):
        rows.append(f"{r.component} earliest {r.earliest_shanghai} (anomalies={int(r.count)})")
    parts.append("Trace anomalies earliest per component: " + "; ".join(rows) + ".")

# Short recommendation
parts.append("Recommendation: prioritize investigation of Mysql01 (many metric faults) and MG01/MG02/IG01/IG02 and Tomcat nodes (early and numerous trace anomalies).")

summary = " ".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  component  earliest_timestamp        earliest_shanghai  count
0      MG01          1616675400  2021-03-25 20:30:00 CST      2
1      MG02          1616675400  2021-03-25 20:30:00 CST      2
2   Mysql01          1616675400  2021-03-25 20:30:00 CST     16,   component  earliest_timestamp        earliest_shanghai  count
0      IG01          1616675400  2021-03-25 20:30:00 CST     61
1      MG01          1616675400  2021-03-25 20:30:00 CST    167
2      MG02          1616675400  2021-03-25 20:30:00 CST    160
3  Tomcat04          1616675400  2021-03-25 20:30:00 CST     65
4  Tomcat03          1616675400  2021-03-25 20:30:00 CST     60
5  Tomcat01          1616675460  2021-03-25 20:31:00 CST     65
6      IG02          1616675580  2021-03-25 20:33:00 CST     60
7  Tomcat02          1616675580  2021-03-25 20:33:00 CST     60)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: incident_trace_df, incident_log_df, trace_thresh, tz
try:
    incident_trace_df
    incident_log_df
    trace_thresh
    tz
except NameError:
    # defensive reload if necessary
    tz = pytz.timezone('Asia/Shanghai')
    incident_trace_df = pd.read_csv('trace.csv')
    incident_log_df = pd.read_csv('log.csv')
    # compute global trace thresholds if missing
    trace_thresh = incident_trace_df.groupby(['cmdb_id','trace_name'])['value'].quantile([0.05,0.95]).unstack(level=-1).reset_index().rename(columns={0.05:'p05',0.95:'p95'})

# 1) Trace groups mentioning Mysql or exact to_Mysql01/from_Mysql01
mask_mysql = incident_trace_df['trace_name'].str.contains('Mysql', na=False) | incident_trace_df['trace_name'].str.contains('to_Mysql01', na=False) | incident_trace_df['trace_name'].str.contains('from_Mysql01', na=False)
trace_mysql_rows = incident_trace_df[mask_mysql].copy()

if trace_mysql_rows.empty:
    trace_to_mysql_df = pd.DataFrame(columns=[
        'component','trace_name','count','anomaly_count','earliest_anomaly_timestamp','earliest_anomaly_shanghai',
        'max_value_in_window','min_value_in_window','max_deviation_ratio'
    ])
else:
    # merge global thresholds
    merged = trace_mysql_rows.merge(trace_thresh, on=['cmdb_id','trace_name'], how='left')
    merged['p95'] = merged['p95'].astype(float)
    merged['p05'] = merged['p05'].astype(float)
    merged['is_high'] = merged['value'] >= merged['p95']
    merged['is_low']  = merged['value'] <= merged['p05']
    merged['is_anomaly'] = merged['is_high'] | merged['is_low']

    def compute_group_stats(g):
        comp = g.name[0]
        tname = g.name[1]
        cnt = int(g.shape[0])
        anom_cnt = int(g['is_anomaly'].sum())
        if anom_cnt > 0:
            earliest_ts = int(g.loc[g['is_anomaly'], 'timestamp'].min())
            earliest_sh = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            earliest_ts = np.nan
            earliest_sh = None
        max_val = float(g['value'].max()) if g['value'].notna().any() else np.nan
        min_val = float(g['value'].min()) if g['value'].notna().any() else np.nan
        p95 = float(g['p95'].iloc[0]) if pd.notna(g['p95'].iloc[0]) else np.nan
        p05 = float(g['p05'].iloc[0]) if pd.notna(g['p05'].iloc[0]) else np.nan

        any_high = g['is_high'].any()
        any_low = g['is_low'].any()

        high_ratio = np.nan
        low_ratio = np.nan
        if any_high and pd.notna(p95):
            high_ratio = np.inf if p95 == 0 else max_val / p95
        if any_low and pd.notna(p05):
            low_ratio = np.inf if min_val == 0 else p05 / min_val

        # fallback: if neither computed but p95 available, use max_val/p95
        ratios = [r for r in [high_ratio, low_ratio] if not pd.isna(r)]
        if not ratios:
            if pd.notna(p95):
                max_dev = np.inf if p95 == 0 else (max_val / p95 if p95 != 0 else np.inf)
            else:
                max_dev = np.nan
        else:
            max_dev = max(ratios)

        return pd.Series({
            'component': comp,
            'trace_name': tname,
            'count': cnt,
            'anomaly_count': anom_cnt,
            'earliest_anomaly_timestamp': earliest_ts,
            'earliest_anomaly_shanghai': earliest_sh,
            'max_value_in_window': round(max_val,6) if not pd.isna(max_val) else np.nan,
            'min_value_in_window': round(min_val,6) if not pd.isna(min_val) else np.nan,
            'max_deviation_ratio': round(max_dev,6) if not pd.isna(max_dev) else np.nan
        })

    grouped = merged.groupby(['cmdb_id','trace_name'], sort=False)
    trace_stats = grouped.apply(compute_group_stats).reset_index(drop=True)

    # sort by earliest_anomaly_timestamp ascending, place NaNs last
    trace_stats_sorted = trace_stats.sort_values(['earliest_anomaly_timestamp','component'], na_position='last').reset_index(drop=True)
    trace_to_mysql_df = trace_stats_sorted.head(20).copy()

# 2) Logs for Mysql01 and Mysql02
mysql_mask = incident_log_df['cmdb_id'].isin(['Mysql01','Mysql02'])
mysql_logs = incident_log_df[mysql_mask].copy()

if mysql_logs.empty:
    mysql_log_summary = pd.DataFrame(columns=['component','total_rows'])
    mysql_log_samples = pd.DataFrame(columns=['timestamp','ts_shanghai','cmdb_id','log_name','value'])
else:
    mysql_log_summary = mysql_logs.groupby('cmdb_id').size().reset_index(name='total_rows').rename(columns={'cmdb_id':'component'}).sort_values('total_rows', ascending=False).reset_index(drop=True)
    # create ts_shanghai
    mysql_logs['ts_shanghai'] = pd.to_datetime(mysql_logs['timestamp'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    mysql_log_samples = mysql_logs[['timestamp','ts_shanghai','cmdb_id','log_name','value']].sort_values('timestamp').head(10).reset_index(drop=True)

# Store variables in kernel as requested: trace_to_mysql_df, mysql_log_samples
trace_to_mysql_df, mysql_log_summary, mysql_log_samples

```
Out[5]:
```
```python
# Summarize results of trace-to-MySQL and MySQL log extraction as a single plain-English string.
summary = (
    "Results for MySQL-related traces and logs in the incident window (2021-03-25 20:30–21:00 UTC+8):\n"
    "- Trace groups to/from or mentioning 'Mysql' : none found in the incident_trace_df (trace_to_mysql_df is empty).\n"
    "- Logs for Mysql01/Mysql02 : no rows found in the incident_log_df for Mysql01 or Mysql02 (mysql_log_samples and mysql_log_summary are empty).\n"
    "Conclusion: within the filtered incident-window trace and log tables there is no direct trace or log evidence referencing MySQL hosts. "
    "Note that metric-based faults for Mysql01 were previously detected (metric telemetry), but corresponding trace or log records for MySQL were not present in the incident-window slices."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [component, trace_name, count, anomaly_count, earliest_anomaly_timestamp, earliest_anomaly_shanghai, max_value_in_window, min_value_in_window, max_deviation_ratio]
Index: [], Empty DataFrame
Columns: [component, total_rows]
Index: [], Empty DataFrame
Columns: [timestamp, ts_shanghai, cmdb_id, log_name, value]
Index: [])```
```